# Scatter plot with 100 text inputs, checksum difference vs logit difference per sublayer

In [ ]:
# imports and load
from pathlib import Path
import importlib.util
import sys

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForMaskedLM

import os
from pathlib import Path

RESULTS_DIR = Path(os.environ["RESULTS_DIR"])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Saving all outputs to:", RESULTS_DIR)

QE_PATH = Path(
    "/home/lecacca/VSCode Environment/BERT-Testing/scripts/quantize_sublayer/quantize_experiments.py"
)

spec = importlib.util.spec_from_file_location("quantize_experiments", QE_PATH)
qe = importlib.util.module_from_spec(spec)
sys.modules["quantize_experiments"] = qe
spec.loader.exec_module(qe)

print("Loaded:", qe)

In [ ]:
# configuration
MODEL_NAME = "/home/lecacca/VSCode Environment/BERT-Testing/bert_uncased_L-2_H-128_A-2"

BLOCK_INDEX = 0
NUM_BITS = 8

SUBLAYERS = [
    "attention.self.query",
    "attention.self.key",
    "attention.self.value",
    "attention.output.dense",
    "intermediate.dense",
    "output.dense",
]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

In [ ]:
# define or load 100 text inputs

# find text samples from online and then import them here as a text file

TEXT_FILE = Path("/home/lecacca/VSCode Environment/BERT-Testing/visualization/masked_texts.txt")

texts = [
    line.strip()
    for line in TEXT_FILE.read_text().splitlines()
    if line.strip()
]

texts = texts[:100]

print("Number of texts:", len(texts))

In [ ]:
# Helper functions

def load_baseline_and_quant_model_for_sublayer(
    model_name,
    sub_layer,
    block_index=0,
    num_bits=8,
    device=torch.device("cpu"),
):

    baseline_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)
    quant_model = AutoModelForMaskedLM.from_pretrained(model_name).to(device)

    baseline_model.eval()
    quant_model.eval()

    baseline_block = baseline_model.bert.encoder.layer[block_index]
    quant_block = quant_model.bert.encoder.layer[block_index]

    baseline_sublayer = qe.get_module_by_name(baseline_block, sub_layer)
    quant_sublayer = qe.get_module_by_name(quant_block, sub_layer)

    qe.quantize_linear_weights(quant_sublayer, num_bits=num_bits)

    return baseline_model, quant_model, baseline_sublayer, quant_sublayer


def get_mask_index(inputs, tokenizer):
    mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=False)

    if mask_positions.shape[0] != 1:
        raise ValueError(
            f"Expected exactly one [MASK], found {mask_positions.shape[0]}"
        )

    return mask_positions[0,1].item()

def run_model_and_capture_module(model, module, inputs, tokenizer):
    saved = {}

    def hook_fn(module, input, output):
        saved["input"] = input[0].detach().cpu()
        saved["output"] = output.detach().cpu()

    handle = module.register_forward_hook(hook_fn)

    with torch.no_grad():
        outputs = model(**inputs)

    handle.remove()

    mask_index = get_mask_index(inputs, tokenizer)
    mask_logits = outputs.logits[0, mask_index, :].detach().cpu()

    return{
        "mask_logits": mask_logits,
        "module_input": saved["input"],
        "module_output": saved["output"],
    }

def compute_metrics_for_one_text(
    text,
    tokenizer,
    baseline_model,
    quant_model,
    baseline_sublayer,
    quant_sublayer,
    device = torch.device("cpu"),
):
    inputs = tokenizer(text, return_tensors = "pt").to(device)

    baseline_run = run_model_and_capture_module(
        baseline_model,
        baseline_sublayer,
        inputs,
        tokenizer,
    )

    quant_run = run_model_and_capture_module(
        quant_model,
        quant_sublayer,
        inputs,
        tokenizer,
    )

    baseline_logits = baseline_run["mask_logits"]
    quant_logits = quant_run["mask_logits"]

    logit_abs_diff = (baseline_logits - quant_logits).abs()

    max_abs_logit_diff = logit_abs_diff.max().item()
    mean_abs_logit_diff = logit_abs_diff.mean().item()
    sum_abs_logit_diff = logit_abs_diff.sum().item()

    baseline_output = baseline_run["module_output"]
    quant_output = quant_run["module_output"]

    baseline_actual_checksum = baseline_output.reshape(-1).sum().item()
    quant_actual_checksum = quant_output.reshape(-1).sum().item()

    actual_checksum_signed_diff = quant_actual_checksum - baseline_actual_checksum
    actual_checksum_abs_diff = abs(actual_checksum_signed_diff)

    # Recompute expected checksum using the same baseline input X.
    # Move module weights to CPU because captured hook tensors are on CPU.
    X = baseline_run["module_input"]

    baseline_weight = baseline_sublayer.weight.detach().cpu()
    baseline_bias = (
        baseline_sublayer.bias.detach().cpu()
        if baseline_sublayer.bias is not None
        else None
    )

    quant_weight = quant_sublayer.weight.detach().cpu()
    quant_bias = (
        quant_sublayer.bias.detach().cpu()
        if quant_sublayer.bias is not None
        else None
    )

    baseline_expected = qe.full_recompute_expected_output_checksum(
        X,
        baseline_weight,
        baseline_bias,
        label="baseline",
    )

    quant_expected = qe.full_recompute_expected_output_checksum(
        X,
        quant_weight,
        quant_bias,
        label="quantized",
    )

    baseline_expected_checksum = baseline_expected["expected_checksum"]
    quant_expected_checksum = quant_expected["expected_checksum"]

    expected_checksum_signed_diff = (
        quant_expected_checksum - baseline_expected_checksum
    )
    expected_checksum_abs_diff = abs(expected_checksum_signed_diff)

    expected_output_l1 = baseline_expected["expected_output_l1_magnitude"]

    expected_checksum_rel_to_l1 = (
        expected_checksum_abs_diff / (expected_output_l1 + 1e-12)
    )

    return {
        "text": text,
        "max_abs_logit_diff": max_abs_logit_diff,
        "mean_abs_logit_diff": mean_abs_logit_diff,
        "sum_abs_logit_diff": sum_abs_logit_diff,
        "baseline_actual_checksum": baseline_actual_checksum,
        "quant_actual_checksum": quant_actual_checksum,
        "actual_checksum_signed_diff": actual_checksum_signed_diff,
        "actual_checksum_abs_diff": actual_checksum_abs_diff,
        "baseline_expected_checksum": baseline_expected_checksum,
        "quant_expected_checksum": quant_expected_checksum,
        "expected_checksum_signed_diff": expected_checksum_signed_diff,
        "expected_checksum_abs_diff": expected_checksum_abs_diff,
        "expected_checksum_rel_to_l1": expected_checksum_rel_to_l1,
    }

In [ ]:
# run all experiments

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

all_rows = []

for sub_layer in SUBLAYERS:
    print(f"\n=== Quantizing sublayer: {sub_layer} ===")

    (
        baseline_model,
        quant_model,
        baseline_sublayer,
        quant_sublayer,
    ) = load_baseline_and_quant_model_for_sublayer(
        MODEL_NAME,
        sub_layer,
        block_index=BLOCK_INDEX,
        num_bits=NUM_BITS,
        device=DEVICE,
    )

    for text_index, text in enumerate(texts):
        if text_index % 10 == 0:
            print(f"  Processing text {text_index + 1}/{len(texts)}")

        metrics = compute_metrics_for_one_text(
            text=text,
            tokenizer=tokenizer,
            baseline_model=baseline_model,
            quant_model=quant_model,
            baseline_sublayer=baseline_sublayer,
            quant_sublayer=quant_sublayer,
            device=DEVICE,
        )

        metrics["text_index"] = text_index
        metrics["sub_layer"] = sub_layer
        metrics["block_index"] = BLOCK_INDEX
        metrics["num_bits"] = NUM_BITS

        all_rows.append(metrics)

    # Free memory before loading next pair of models.
    del baseline_model
    del quant_model
    torch.cuda.empty_cache()

df = pd.DataFrame(all_rows)

print("Done.")
print("Rows:", len(df))
df.head()

In [ ]:
# add relative checksum drift columns

eps = 1e-12

# Relative to baseline expected checksum:
# |quant_checksum - baseline_checksum| / |baseline_checksum|
df["expected_checksum_rel_to_baseline"] = (
    df["expected_checksum_abs_diff"] / (df["baseline_expected_checksum"].abs() + eps)
)

# Relative to expected output L1 magnitude:
# |quant_checksum - baseline_checksum| / sum(|baseline expected output|)
# This may already exist from compute_metrics_for_one_text, but this keeps it safe.
if "expected_checksum_rel_to_l1" not in df.columns:
    df["expected_checksum_rel_to_l1"] = (
        df["expected_checksum_abs_diff"] / (df["baseline_expected_output_l1_magnitude"] + eps)
    )

df[
    [
        "sub_layer",
        "mean_abs_logit_diff",
        "max_abs_logit_diff",
        "expected_checksum_abs_diff",
        "expected_checksum_rel_to_baseline",
        "expected_checksum_rel_to_l1",
    ]
].head()

df.to_csv(
    RESULTS_DIR / "quantization_sublayer_100_text_results_with_relative_metrics.csv",
    index=False,
)

In [ ]:
# four panel scatterplot

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(
    data=df,
    x="mean_abs_logit_diff",
    y="expected_checksum_rel_to_baseline",
    hue="sub_layer",
    alpha=0.75,
    s=45,
    ax=axes[0, 0],
)

axes[0, 0].set_title("Relative to baseline vs mean logit drift")
axes[0, 0].set_xlabel("Mean absolute [MASK] logit difference")
axes[0, 0].set_ylabel("Expected checksum drift / |baseline checksum|")
axes[0, 0].grid(True, alpha=0.3)

sns.scatterplot(
    data=df,
    x="mean_abs_logit_diff",
    y="expected_checksum_rel_to_l1",
    hue="sub_layer",
    alpha=0.75,
    s=45,
    ax=axes[0, 1],
)

axes[0, 1].set_title("Relative to L1 magnitude vs mean logit drift")
axes[0, 1].set_xlabel("Mean absolute [MASK] logit difference")
axes[0, 1].set_ylabel("Expected checksum drift / expected output L1")
axes[0, 1].grid(True, alpha=0.3)

sns.scatterplot(
    data=df,
    x="max_abs_logit_diff",
    y="expected_checksum_rel_to_baseline",
    hue="sub_layer",
    alpha=0.75,
    s=45,
    ax=axes[1, 0],
)

axes[1, 0].set_title("Relative to baseline vs max logit drift")
axes[1, 0].set_xlabel("Max absolute [MASK] logit difference")
axes[1, 0].set_ylabel("Expected checksum drift / |baseline checksum|")
axes[1, 0].grid(True, alpha=0.3)

sns.scatterplot(
    data=df,
    x="max_abs_logit_diff",
    y="expected_checksum_rel_to_l1",
    hue="sub_layer",
    alpha=0.75,
    s=45,
    ax=axes[1, 1],
)

axes[1, 1].set_title("Relative to L1 magnitude vs max logit drift")
axes[1, 1].set_xlabel("Max absolute [MASK] logit difference")
axes[1, 1].set_ylabel("Expected checksum drift / expected output L1")
axes[1, 1].grid(True, alpha=0.3)

# Remove duplicate legends from individual axes
for ax in axes.flatten():
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

# Add one shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    title="Quantized sublayer",
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
)

fig.suptitle(
    f"Relative Checksum Drift vs Logit Drift\n"
    f"Block {BLOCK_INDEX}, {NUM_BITS}-bit fake quantization",
    fontsize=14,
)

plt.tight_layout(rect=[0, 0, 0.85, 0.95])

plt.savefig(
    RESULTS_DIR / "relative_checksum_vs_logit_drift_4panel.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# inspect/save results

df.head()

df.groupby("sub_layer")[
    [
        "max_abs_logit_diff",
        "mean_abs_logit_diff",
        "expected_checksum_abs_diff",
        "actual_checksum_abs_diff",
        "expected_checksum_rel_to_l1",
    ]
].describe()

OUTPUT_CSV = Path("quantization_sublayer_100_text_results.csv")
df.to_csv(RESULTS_DIR / OUTPUT_CSV, index=False)
print("Saved to:", OUTPUT_CSV.resolve())

In [ ]:
# scatterplot using expected checksum difference

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=df,
    x="mean_abs_logit_diff",
    y="expected_checksum_abs_diff",
    hue="sub_layer",
    alpha=0.75,
    s=55,
)

plt.xlabel("Mean absolute [MASK] logit difference")
plt.ylabel("Expected checksum absolute difference")
plt.title(
    f"Checksum Drift vs Logit Drift\n"
    f"Block {BLOCK_INDEX}, {NUM_BITS}-bit fake quantization"
)

plt.grid(True, alpha=0.3)
plt.legend(title="Quantized sublayer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# scatterplot using max logit difference

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=df,
    x="max_abs_logit_diff",
    y="expected_checksum_abs_diff",
    hue="sub_layer",
    alpha=0.75,
    s=55,
)

plt.xlabel("Max absolute [MASK] logit difference")
plt.ylabel("Expected checksum absolute difference")
plt.title(
    f"Checksum Drift vs Max Logit Drift\n"
    f"Block {BLOCK_INDEX}, {NUM_BITS}-bit fake quantization"
)

plt.grid(True, alpha=0.3)
plt.legend(title="Quantized sublayer", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# one subplot per sublayer

g = sns.FacetGrid(
    df,
    col="sub_layer",
    col_wrap=3,
    height=4,
    sharex=False,
    sharey=False,
)

g.map_dataframe(
    sns.scatterplot,
    x="mean_abs_logit_diff",
    y="expected_checksum_abs_diff",
    alpha=0.75,
    s=45,
)

g.set_axis_labels(
    "Mean absolute [MASK] logit difference",
    "Expected checksum absolute difference",
)

g.fig.suptitle(
    f"Checksum Drift vs Logit Drift by Sublayer\n"
    f"Block {BLOCK_INDEX}, {NUM_BITS}-bit fake quantization",
    y=1.03,
)

for ax in g.axes.flatten():
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()